In [1]:
import pandas as pd

df = pd.read_csv("heart.csv")


In [2]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])


In [9]:
df['risk_factor_count'] = (
    (df['RestingBP'] > 130).astype(int) +
    (df['Cholesterol'] > 250).astype(int) +
    (df['FastingBS'] == 1).astype(int) +
    (df['Age'] > 55).astype(int)
)


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [11]:
X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [12]:
X_train

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,risk_factor_count
795,42,1,2,120,240,1,1,194,0,0.8,0,1
25,36,1,2,130,209,0,1,178,0,0.0,2,0
84,56,1,0,150,213,1,1,125,1,1.0,1,3
10,37,0,2,130,211,0,1,142,0,0.0,2,0
344,51,1,0,120,0,1,1,104,0,0.0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...
106,48,0,0,120,254,0,2,110,0,0.0,2,1
270,45,1,0,120,225,0,1,140,0,0.0,2,0
860,60,1,0,130,253,0,1,144,1,1.4,2,2
435,60,1,0,152,0,0,2,118,1,0.0,2,2


In [13]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [15]:
X_train.shape

(734, 12)

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score


In [19]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    # "Random Forest": RandomForestClassifier(n_estimators=200),
    # "KNN": KNeighborsClassifier(n_neighbors=7),
    # "SVM": SVC(probability=True)
}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]
    print(f"{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("ROC AUC:", roc_auc_score(y_test, y_prob))
    print("-"*30)


Logistic Regression
Accuracy: 0.8478260869565217
ROC AUC: 0.9104260225755553
------------------------------


In [89]:
age = 32
sex = 1              # 1 = Male
cp = 2               # ChestPainType (encoded)
bp = 118
chol = 180
fbs = 0
ecg = 1              # RestingECG (encoded)
maxhr = 175
angina = 0           # 0 = No
oldpeak = 0.0
slope = 2            # ST_Slope (encoded)

# Calculate risk factor count
risk_factor_count = (
    (bp > 130) +
    (chol > 250) +
    (fbs == 1) +
    (age > 55)
)

# Create input array (ORDER MATTERS!)
input_data = np.array([[age, sex, cp, bp, chol, fbs, ecg,
                         maxhr, angina, oldpeak, slope,
                         risk_factor_count]])

# Scale input
input_scaled = scaler.transform(input_data)

# Prediction
prediction = model.predict(input_scaled)
probability = model.predict_proba(input_scaled)

print("Prediction:", prediction)
print("Probability:", probability)

if prediction[0] == 1:
    print("❌ High Risk of Heart Disease")
else:
    print("✅ Low Risk of Heart Disease")


Prediction: [0]
Probability: [[0.92970946 0.07029054]]
✅ Low Risk of Heart Disease


c:\Users\Dell\Desktop\new_heart_data\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [90]:
age = 63
sex = 1
cp = 0
bp = 155
chol = 285
fbs = 1
ecg = 2
maxhr = 120
angina = 1
oldpeak = 2.5
slope = 0

risk_factor_count = (
    (bp > 130) +
    (chol > 250) +
    (fbs == 1) +
    (age > 55)
)

input_data = np.array([[age, sex, cp, bp, chol, fbs, ecg,
                         maxhr, angina, oldpeak, slope,
                         risk_factor_count]])

input_scaled = scaler.transform(input_data)

prediction = model.predict(input_scaled)
probability = model.predict_proba(input_scaled)

print("Prediction:", prediction)
print("Probability:", probability)

if prediction[0] == 1:
    print("❌ High Risk of Heart Disease")
else:
    print("✅ Low Risk of Heart Disease")


Prediction: [1]
Probability: [[0.10642873 0.89357127]]
❌ High Risk of Heart Disease


c:\Users\Dell\Desktop\new_heart_data\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [91]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Logistic Regression
log_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])
log_pipe.fit(X_train, y_train)
joblib.dump(log_pipe, open("models/logistic.pkl", "wb"))

# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
joblib.dump(rf, open("models/random_forest.pkl", "wb"))

# KNN
knn_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier(n_neighbors=7))
])
knn_pipe.fit(X_train, y_train)
joblib.dump(knn_pipe, open("models/knn.pkl", "wb"))

# SVM (BEST ROC-AUC)
svm_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(probability=True))
])
svm_pipe.fit(X_train, y_train)
joblib.dump(svm_pipe, open("models/svm.pkl", "wb"))


In [81]:
input_df = pd.DataFrame([{
    "Age": age,
    "Sex": sex,
    "ChestPainType": chest_pain,
    "RestingBP": bp,
    "Cholesterol": chol,
    "FastingBS": fbs,
    "RestingECG": ecg,
    "MaxHR": maxhr,
    "ExerciseAngina": angina,
    "Oldpeak": oldpeak,
    "ST_Slope": slope,
    "risk_factor_count": risk_factor_count
}])


NameError: name 'chest_pain' is not defined

In [92]:
model = joblib.load(open("models/svm.pkl", "rb"))

model.predict(input_df)
model.predict_proba(input_df)


c:\Users\Dell\Desktop\new_heart_data\venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
c:\Users\Dell\Desktop\new_heart_data\venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


array([[0.31045088, 0.68954912]])